# 14. 시퀀스 레이블링 (Sequence Labeling)

이번 장에서는 파이토치로 인공 신경망을 이용하여 **태깅(tagging) 작업**을 하는 모델을 만든다. 대표적인 태깅 작업인 **개체명 인식(NER)**과 **품사 태깅**은 두 가지 공통점을 가진다. 하나는 RNN의 **다-대-다(Many-to-Many)** 작업이라는 점이고, 다른 하나는 앞·뒤 시점의 입력을 모두 참고하는 **양방향 RNN(Bidirectional RNN)**을 사용한다는 점이다. 개념을 먼저 정리한 뒤, 양방향 LSTM으로 **개체명 인식 모델**을 처음부터 끝까지 구현한다.

> **이 장의 흐름** : 시퀀스 레이블링 개념(X·y 병렬 쌍) → 양방향 RNN과 다-대-다 구조 → 개체명 인식(NER)이란 → NLTK로 NER 맛보기 → 양방향 LSTM NER 실습(데이터 로드 → Vocab → 정수 인코딩 → 패딩 → 모델링 → 학습 → 평가 → 인퍼런스).

> 실습 데이터는 영어 개체명 인식 코퍼스(CoNLL 2003)로, 저자의 깃허브에서 바로 내려받아 사용한다. 형태소 분석기가 필요 없는 영어 데이터이므로 **다운로드만 되면 그 자리에서 전 과정을 실행**할 수 있다.

# 1. 시퀀스 레이블링(Sequence Labeling)이란?

**시퀀스 레이블링**은 입력 시퀀스의 **각 원소마다 레이블을 하나씩 붙이는** 작업이다. 문장을 이루는 단어 하나하나에 품사나 개체명 같은 태그를 부여하는 **태깅 작업**이 대표적이다. 이 장에서 만드는 개체명 인식기와 품사 태거가 모두 여기에 속한다.

태깅 작업은 앞서 배운 텍스트 분류와 마찬가지로 **지도 학습(Supervised Learning)**이다. 태깅할 단어 데이터를 **X**, 레이블에 해당하는 태깅 정보를 **y**라고 부르고, 훈련·테스트 데이터를 각각 **X_train / X_test**, **y_train / y_test**로 명명한다.

## 1-1. 훈련 데이터의 이해

이 작업에서 X와 y 데이터의 **쌍(pair)은 병렬 구조**를 가진다. 즉 **X와 y의 각 샘플의 길이가 같다.** 품사 태깅을 예로 들어 X_train과 y_train의 샘플 4개를 보면 다음과 같은 구조가 된다.

| idx | X_train | y_train | 길이 |
|:--:|:--|:--|:--:|
| 0 | ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb'] | ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O'] | 8 |
| 1 | ['peter', 'blackburn'] | ['B-PER', 'I-PER'] | 2 |
| 2 | ['brussels', '1996-08-22'] | ['B-LOC', 'O'] | 2 |
| 3 | ['The', 'European', 'Commission'] | ['O', 'B-ORG', 'I-ORG'] | 3 |

가령 `X_train[3]`의 `'The'`와 `y_train[3]`의 `'O'`는 하나의 쌍이고, `'European'`과 `'B-ORG'`, `'Commission'`과 `'I-ORG'`가 각각 쌍을 이룬다. 이렇게 병렬 관계를 가지는 데이터는 **정수 인코딩**을 거친 뒤, 모든 데이터의 길이를 동일하게 맞추는 **패딩(Padding)** 작업을 거쳐 모델에 입력된다.

## 1-2. 시퀀스 레이블링 작업의 정의

정리하면, 입력 시퀀스 $X = [x_1, x_2, x_3, ..., x_n]$ 에 대하여 레이블 시퀀스 $y = [y_1, y_2, y_3, ..., y_n]$ 를 각각 부여하는 작업을 **시퀀스 레이블링 작업(Sequence Labeling Task)**이라고 한다. 태깅 작업은 이 시퀀스 레이블링의 대표적인 예다.

## 1-3. 양방향 RNN (Bidirectional RNN)

이 장에서도 바닐라 RNN 대신 성능이 개선된 **LSTM**이나 **GRU**를 사용한다. 다만 텍스트 분류에서는 단방향 RNN을 썼던 것과 달리, 태깅에서는 **양방향 RNN**을 사용한다. 어떤 단어의 태그를 정할 때 **이전 시점의 단어뿐 아니라 다음 시점의 단어 정보도 함께 참고**하기 위해서다. 예컨대 어떤 단어가 조직명인지 사람 이름인지는 앞뒤 문맥을 모두 봐야 정확히 판단할 수 있다.

양방향은 기존 단방향 RNN 생성자에 `bidirectional=True` 인자만 넘기면 된다.

```python
nn.RNN(input_size=input_size, hidden_size=hidden_size, num_layers=1,
       batch_first=True, bidirectional=True)
```

## 1-4. RNN의 다-대-다(Many-to-Many) 문제

태깅은 입력의 매 시점마다 출력이 나오는 **다-대-다(Many-to-Many)** 구조다. 예를 들어 `X_train[0] = ['EU', 'rejects', 'German', 'call']`을 4개 시점까지 진행하면, 각 시점 $x_t$의 입력에 대해 그 자리의 태그 $y_t$가 하나씩 출력된다.

| 시점 | $x_1$ (EU) | $x_2$ (rejects) | $x_3$ (German) | $x_4$ (call) |
|:--:|:--:|:--:|:--:|:--:|
| 출력 | $y_1$ = B-ORG | $y_2$ = O | $y_3$ = B-MISC | $y_4$ = O |

**단방향 RNN**이라면 각 시점의 은닉 상태는 **왼쪽(과거) 정보만** 담아 다음 시점으로 전달된다. 반면 **양방향 RNN**은 시퀀스를 정방향으로 한 번, 역방향으로 한 번 읽어, 각 시점에서 **순방향 은닉 상태와 역방향 은닉 상태를 함께 사용**한다. 그래서 각 시점의 출력층은 앞뒤 양쪽 문맥을 모두 반영한 표현을 입력으로 받게 된다. 이것이 태깅 작업에서 양방향 구조를 쓰는 이유다.

# 2. 개체명 인식 (Named Entity Recognition)

시퀀스 레이블링의 대표 작업인 **개체명 인식**을 구체적으로 살펴본다.

## 2-1. 개체명 인식이란?

**개체명 인식(Named Entity Recognition)**이란 말 그대로 **이름을 가진 개체(named entity)를 인식**하는 것이다. 좀 더 쉽게 말하면, 어떤 이름을 뜻하는 단어를 보고 그 단어가 **어떤 유형(사람·장소·조직 등)인지**를 판별하는 작업이다.

예를 들어 `"유정이는 2018년에 골드만삭스에 입사했다."`라는 문장에 대해 사람·조직·시간 개체명 인식을 수행하면 다음과 같은 결과가 나온다.

| 단어 | 개체명 유형 |
|:--|:--|
| 유정 | 사람(PERSON) |
| 2018년 | 시간(TIME) |
| 골드만삭스 | 조직(ORGANIZATION) |

## 2-2. NLTK를 이용한 개체명 인식

**NLTK**는 개체명 인식기(NER chunker)를 내장하고 있어, 별도로 모델을 만들지 않아도 개체명 인식을 바로 수행할 수 있다. 개체명을 태깅하려면 먼저 **품사 태깅(pos_tag)**이 되어 있어야 하며, 그 결과를 `ne_chunk`에 넣으면 된다.

> 실행 시 `maxent_ne_chunker`, `words` 등의 설치를 요구하는 에러가 뜨면 지시대로 다운로드하면 된다. 아래 셀에서는 필요한 데이터를 미리 내려받는다. (NLTK 버전에 따라 `_tab`, `_eng`가 붙은 최신 패키지명도 함께 받는다.)

In [ ]:
import nltk

# NER에 필요한 NLTK 데이터 다운로드 (버전에 따라 구/신 패키지명을 모두 받는다)
for pkg in ['punkt', 'punkt_tab',
            'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
            'maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words']:
    nltk.download(pkg, quiet=True)

from nltk import word_tokenize, pos_tag, ne_chunk

sentence = "James is working at Disney in London"

# 1) 토큰화 후 품사 태깅
tokenized_sentence = pos_tag(word_tokenize(sentence))
print(tokenized_sentence)

# 2) 품사 태깅 결과를 개체명 인식기에 입력
ner_sentence = ne_chunk(tokenized_sentence)
print(ner_sentence)
# James -> PERSON, Disney -> ORGANIZATION, London -> GPE(위치) 로 인식된다.

# 3. 양방향 LSTM으로 개체명 인식 모델 만들기

이제 인공 신경망으로 개체명 인식 모델을 직접 만든다. **2층짜리 양방향 LSTM**을 사용하며, 데이터 로드부터 전처리(Vocab·정수 인코딩·패딩), 모델링, 학습, 평가, 인퍼런스까지 전 과정을 구현한다.

## 3-1. 데이터 로드 및 단어 토큰화

NLTK 코퍼스 기반으로 이미 **토큰화와 품사·개체명 태깅이 완료된** 영어 문장 데이터를 저자의 깃허브에서 내려받는다. 파일의 각 줄은 `단어 품사 청크 개체명` 형태이며, 빈 줄(또는 `-DOCSTART-`)이 문장의 경계가 된다. 우리는 이 중 **단어와 개체명 태그만** 뽑아 문장 단위로 모은다.

In [ ]:
import urllib.request
import numpy as np
from tqdm import tqdm
import re
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# 저자의 깃허브에서 CoNLL 개체명 인식 데이터 다운로드
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ukairia777/tensorflow-nlp-tutorial/main/12.%20RNN%20Sequence%20Labeling/dataset/train.txt",
    filename="train.txt")

f = open('train.txt', 'r')
tagged_sentences = []
sentence = []
for line in f:
    # 빈 줄이나 -DOCSTART- 는 문장 경계이므로, 지금까지 모은 문장을 저장한다.
    if len(line) == 0 or line.startswith('-DOCSTART') or line[0] == "\n":
        if len(sentence) > 0:
            tagged_sentences.append(sentence)
            sentence = []
        continue
    splits = line.split(' ')                    # 공백으로 속성 구분
    splits[-1] = re.sub(r'\n', '', splits[-1])  # 마지막 속성의 줄바꿈 제거
    word = splits[0].lower()                    # 단어는 소문자로 통일
    sentence.append([word, splits[-1]])         # [단어, 개체명 태그]만 기록

print("전체 샘플 개수: ", len(tagged_sentences))

첫 번째 샘플을 출력해 구조를 확인한다. 각 원소가 `[단어, 개체명 태그]` 쌍으로 되어 있음을 볼 수 있다. 이어서 예측에 사용할 **X(단어)**와 예측 대상인 **y(개체명 태그)**를 분리한다.

In [ ]:
print(tagged_sentences[0])  # 첫 번째 샘플

# 각 샘플에서 단어는 sentences(X), 태그는 ner_tags(y)로 분리
sentences, ner_tags = [], []
for tagged_sentence in tagged_sentences:
    sentence, tag_info = zip(*tagged_sentence)  # (단어들), (태그들)로 분해
    sentences.append(list(sentence))
    ner_tags.append(list(tag_info))

print(sentences[0])
print(ner_tags[0])
print()
# 길이가 다른 12번 샘플도 확인 (샘플마다 길이가 다르다)
print(sentences[12])
print(ner_tags[12])

이제 훈련 데이터, 테스트 데이터, 그리고 학습 중 성능을 확인할 검증 데이터로 나눈다. `random_state`를 고정해 분할이 재현되도록 한다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    sentences, ner_tags, test_size=.2, random_state=777)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=.2, random_state=777)

print('훈련 데이터의 개수 :', len(X_train))
print('검증 데이터의 개수 :', len(X_valid))
print('테스트 데이터의 개수 :', len(X_test))

# 상위 샘플 2개 출력 (현재 단어 토큰화만 된 상태)
print('-' * 30)
for sent in X_train[:2]:
    print(sent)

## 3-2. 단어 집합(Vocab) 만들기

각 단어의 등장 빈도를 세어 **단어 집합**을 만든다. `Counter`로 빈도를 집계하고, 빈도가 높은 순으로 정렬해 인덱스를 부여할 것이다.

In [ ]:
# 훈련 데이터의 모든 단어를 모아 빈도 집계
word_list = []
for sent in X_train:
    for word in sent:
        word_list.append(word)
word_counts = Counter(word_list)
print('총 단어수 :', len(word_counts))

# 특정 단어의 등장 횟수 확인
print('단어 the 의 등장 횟수 :', word_counts['the'])
print('단어 love 의 등장 횟수 :', word_counts['love'])

# 빈도 상위 순으로 정렬
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
print('등장 빈도 상위 10개 :', vocab[:10])

## 3-3. 정수 인코딩

단어 집합을 실제 정수로 매핑한다. 이때 **패딩용 토큰 `<PAD>`에 0**, **미등록 단어(OOV) 처리용 토큰 `<UNK>`에 1**을 먼저 할당하고, 나머지 단어에는 빈도 순으로 2번부터 인덱스를 부여한다.

In [ ]:
word_to_index = {}
word_to_index['<PAD>'] = 0   # 패딩 토큰
word_to_index['<UNK>'] = 1   # OOV(미등록 단어) 토큰
for index, word in enumerate(vocab):
    word_to_index[word] = index + 2

vocab_size = len(word_to_index)
print('단어 집합의 크기 :', vocab_size)
print('<PAD> ->', word_to_index['<PAD>'])
print('<UNK> ->', word_to_index['<UNK>'])
print('the   ->', word_to_index['the'])

텍스트를 정수 시퀀스로 바꾸는 함수를 만든다. 단어 집합에 없는 단어(OOV)를 만나면 `<UNK>`에 해당하는 정수 **1**로 변환한다. 이 함수를 훈련·검증·테스트 데이터에 모두 적용한다.

In [ ]:
def texts_to_sequences(tokenized_X_data, word_to_index):
    encoded_X_data = []
    for sent in tokenized_X_data:
        index_sequences = []
        for word in sent:
            try:
                index_sequences.append(word_to_index[word])
            except KeyError:
                index_sequences.append(word_to_index['<UNK>'])  # OOV -> 1
        encoded_X_data.append(index_sequences)
    return encoded_X_data

encoded_X_train = texts_to_sequences(X_train, word_to_index)
encoded_X_valid = texts_to_sequences(X_valid, word_to_index)
encoded_X_test = texts_to_sequences(X_test, word_to_index)

# 정수 인코딩된 상위 2개 샘플
for sent in encoded_X_train[:2]:
    print(sent)

정수를 다시 단어로 되돌리려면 `word_to_index`의 key와 value를 뒤집은 `index_to_word`를 만들면 된다. 첫 번째 샘플을 복원해 인코딩이 올바른지 확인한다.

이어서 **레이블(개체명 태그)에 대해서도 정수 인코딩**을 준비한다. 먼저 레이블에 존재하는 모든 태그의 집합을 구한다.

In [ ]:
# 정수 -> 단어 매핑 (예측 결과 복원용)
index_to_word = {}
for key, value in word_to_index.items():
    index_to_word[value] = key

decoded_sample = [index_to_word[word] for word in encoded_X_train[0]]
print('기존 첫 번째 샘플 :', X_train[0])
print('복원된 첫 번째 샘플 :', decoded_sample)

# y_train에 존재하는 모든 태그의 집합
flatten_tags = [tag for sent in y_train for tag in sent]
tag_vocab = list(set(flatten_tags))
print('태그 집합 :', tag_vocab)
print('태그 집합의 크기 :', len(tag_vocab))

레이블의 각 태그에 정수를 부여한다. 개체명 인식은 **다-대-다(Many-to-Many)** 문제라 레이블도 시퀀스이므로, 레이블 역시 정수 **시퀀스**로 변환해야 한다. 여기서도 패딩용 `<PAD>`에 0을 할당하고 나머지 태그에 1번부터 부여한다.

> **참고** : `list(set(...))`은 순서를 보장하지 않으므로, 각 태그에 붙는 정수는 실행할 때마다 달라질 수 있다. 다만 태그와 정수의 대응이 한 번의 실행 안에서는 일관되므로 **학습 결과에는 영향이 없다.** 아래는 예시 매핑이며 실제 값은 다를 수 있다.
>
> `{'<PAD>': 0, 'B-PER': 1, 'I-MISC': 2, 'B-ORG': 3, 'I-PER': 4, 'B-LOC': 5, 'I-LOC': 6, 'I-ORG': 7, 'O': 8, 'B-MISC': 9}`

레이블도 앞서 만든 `texts_to_sequences`를 그대로 재사용해 정수 인코딩한다. (교안은 레이블 전용 함수도 소개하지만, 동작이 같으므로 하나로 통일한다.)

In [ ]:
tag_to_index = {}
tag_to_index['<PAD>'] = 0
for index, word in enumerate(tag_vocab):
    tag_to_index[word] = index + 1

tag_vocab_size = len(tag_to_index)
print('태그 집합 :', tag_to_index)
print('패딩 포함 태그 집합의 크기 :', tag_vocab_size)

# 레이블도 정수 시퀀스로 인코딩
encoded_y_train = texts_to_sequences(y_train, tag_to_index)
encoded_y_valid = texts_to_sequences(y_valid, tag_to_index)
encoded_y_test = texts_to_sequences(y_test, tag_to_index)

# 상위 2개 샘플의 인코딩 결과
for tags in encoded_y_train[:2]:
    print(tags)

## 3-4. 패딩 (Padding)

모델은 길이가 동일한 입력을 받아야 하므로, 모든 샘플의 길이를 특정 값 `max_len`으로 맞추는 **패딩**이 필요하다. 적절한 `max_len`을 정하기 위해 먼저 샘플 길이의 분포를 확인한다.

In [ ]:
print('샘플의 최대 길이 : %d' % max(len(l) for l in encoded_X_train))
print('샘플의 평균 길이 : %f' % (sum(map(len, encoded_X_train)) / len(encoded_X_train)))

plt.hist([len(s) for s in encoded_X_train], bins=50)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

최대 길이가 78이므로, 특정 `max_len` 이하인 샘플이 전체의 몇 %인지 확인하는 함수로 커버리지를 점검한다. 여기서는 최대 길이보다 살짝 큰 **80**으로 패딩한다.

In [ ]:
def below_threshold_len(max_len, nested_list):
    count = 0
    for sentence in nested_list:
        if len(sentence) <= max_len:
            count = count + 1
    print('전체 샘플 중 길이가 %s 이하인 샘플의 비율: %s'
          % (max_len, (count / len(nested_list)) * 100))

max_len = 80
below_threshold_len(max_len, encoded_X_train)

`max_len`보다 짧은 데이터의 **뒤쪽에 0을 채우는** 패딩 함수를 만든다. 개체명 인식 같은 다-대-다 문제에서는 **레이블도 함께 패딩**해야 한다. 훈련·검증·테스트의 X와 y를 모두 패딩한 뒤 크기를 확인한다.

In [ ]:
def pad_sequences(sentences, max_len):
    features = np.zeros((len(sentences), max_len), dtype=int)
    for index, sentence in enumerate(sentences):
        if len(sentence) != 0:
            features[index, :len(sentence)] = np.array(sentence)[:max_len]
    return features

padded_X_train = pad_sequences(encoded_X_train, max_len=max_len)
padded_X_valid = pad_sequences(encoded_X_valid, max_len=max_len)
padded_X_test = pad_sequences(encoded_X_test, max_len=max_len)
padded_y_train = pad_sequences(encoded_y_train, max_len=max_len)
padded_y_valid = pad_sequences(encoded_y_valid, max_len=max_len)
padded_y_test = pad_sequences(encoded_y_test, max_len=max_len)

print('훈련 데이터의 크기 :', padded_X_train.shape)
print('검증 데이터의 크기 :', padded_X_valid.shape)
print('테스트 데이터의 크기 :', padded_X_test.shape)
print('-' * 30)
print('훈련 레이블의 크기 :', padded_y_train.shape)
print('검증 레이블의 크기 :', padded_y_valid.shape)
print('테스트 레이블의 크기 :', padded_y_test.shape)

패딩된 결과를 확인한다. 실제 단어/태그 뒤로 0이 채워져 모든 샘플이 길이 80이 되었음을 볼 수 있다.

In [ ]:
print('훈련 데이터 상위 2개')
print(padded_X_train[:2])
print('-' * 5 + '레이블' + '-' * 5)
print(padded_y_train[:2])

## 3-5. 모델링

이제 모델을 구현한다. 먼저 GPU 사용이 가능한 환경인지 확인한다. `cuda`가 출력되면 GPU를 쓸 수 있다. (Colab에서 `cpu`가 나오면 상단 메뉴에서 **런타임 > 런타임 유형 변경 > 하드웨어 가속기 > GPU**로 변경한다.)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

USE_CUDA = torch.cuda.is_available()
device = torch.device("cuda" if USE_CUDA else "cpu")
print("cpu와 cuda 중 다음 기기로 학습함:", device)

모델 구조를 설계한다. 만약 **단방향 GRU**를 쓴다면 코드는 아래와 같다.

```python
class NERTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(NERTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)      # (batch, seq_len, embedding_dim)
        gru_out, _ = self.gru(embedded)   # (batch, seq_len, hidden_dim)
        logits = self.fc(gru_out)         # (batch, seq_len, output_dim)
        return logits
```

이를 **2층짜리 양방향 LSTM**으로 바꾸려면 네 가지만 수정하면 된다.

- `nn.GRU` → `nn.LSTM` 으로 교체
- `num_layers` 매개변수를 추가해 층 수를 지정 (여기서는 2)
- `bidirectional=True` 를 추가해 양방향으로 설정
- `nn.Linear`의 입력 차원을 `hidden_dim * 2` 로 변경 (양방향이라 순방향+역방향 은닉 상태가 연결되므로)

In [ ]:
class NERTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers=2):
        super(NERTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # 2층 양방향 LSTM
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True)
        # 양방향이므로 은닉 차원의 2배가 입력
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        # x: (batch_size, seq_length)
        embedded = self.embedding(x)       # (batch, seq_len, embedding_dim)
        lstm_out, _ = self.lstm(embedded)  # (batch, seq_len, hidden_dim*2)
        logits = self.fc(lstm_out)         # (batch, seq_len, output_dim)
        return logits

사용할 데이터를 파이토치 텐서로 변환하고, 배치 단위 처리를 위해 `DataLoader`로 감싼다. 훈련 데이터만 셔플하고 검증·테스트는 순서를 유지한다.

In [ ]:
X_train_tensor = torch.tensor(padded_X_train, dtype=torch.long)
y_train_tensor = torch.tensor(padded_y_train, dtype=torch.long)
X_valid_tensor = torch.tensor(padded_X_valid, dtype=torch.long)
y_valid_tensor = torch.tensor(padded_y_valid, dtype=torch.long)
X_test_tensor = torch.tensor(padded_X_test, dtype=torch.long)
y_test_tensor = torch.tensor(padded_y_test, dtype=torch.long)

train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
train_dataloader = torch.utils.data.DataLoader(train_dataset, shuffle=True, batch_size=32)
valid_dataset = torch.utils.data.TensorDataset(X_valid_tensor, y_valid_tensor)
valid_dataloader = torch.utils.data.DataLoader(valid_dataset, shuffle=False, batch_size=32)
test_dataset = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)
test_dataloader = torch.utils.data.DataLoader(test_dataset, shuffle=False, batch_size=32)

print('단어 집합의 크기:', vocab_size)

모델 객체를 만들기 위한 하이퍼파라미터는 다음과 같다. 임베딩 차원 **100**, LSTM 은닉 차원 **256**, 출력 차원은 태그 집합 크기인 **`tag_vocab_size`(=10)**, 학습률 **0.01**, 에포크 **10**, LSTM 층 수 **2**로 지정한다.

비용 함수 `nn.CrossEntropyLoss`에는 `ignore_index` 옵션이 있어, 특정 인덱스에 대한 loss를 제외할 수 있다. **`ignore_index=0`**을 주면 **패딩 위치(0)에 대해서는 loss를 계산하지 않는다.** (또한 `CrossEntropyLoss`는 내부에 소프트맥스를 포함하므로, 모델 출력에 별도로 소프트맥스를 적용하지 않는다.)

In [ ]:
embedding_dim = 100
hidden_dim = 256
output_dim = tag_vocab_size
learning_rate = 0.01
num_epochs = 10
num_layers = 2

# 모델, 손실 함수, 옵티마이저
model = NERTagger(vocab_size, embedding_dim, hidden_dim, output_dim, num_layers)
model.to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # 패딩(0)은 loss 계산에서 제외
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## 3-6. 평가 코드 작성

학습 중에 정확도와 loss를 측정하려면 평가 함수를 먼저 만들어야 한다. 우선 예측값과 실제값으로 정확도를 구하는 `calculate_accuracy`를 작성한다. 핵심은 **패딩 토큰 위치는 정확도 계산에서 제외**한다는 점이다.

In [ ]:
def calculate_accuracy(logits, labels, ignore_index=0):
    predicted = torch.argmax(logits, dim=1)   # 예측 레이블
    mask = (labels != ignore_index)           # 패딩이 아닌 위치만
    correct = (predicted == labels).masked_select(mask).sum().item()
    total = mask.sum().item()
    accuracy = correct / total
    return accuracy

검증 데이터로더로 모델 성능을 측정하는 `evaluate` 함수를 만든다. 내부에서 위의 `calculate_accuracy`를 호출한다. 평가 시에는 `model.eval()`과 `torch.no_grad()`로 그래디언트 계산을 끈다.

In [ ]:
def evaluate(model, valid_dataloader, criterion, device):
    val_loss = 0
    val_correct = 0
    val_total = 0

    model.eval()
    with torch.no_grad():
        for batch_X, batch_y in valid_dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            logits = model(batch_X)  # 순전파
            # (batch*seq_len, output_dim) 형태로 펼쳐 loss 계산
            loss = criterion(logits.view(-1, output_dim), batch_y.view(-1))

            val_loss += loss.item()
            val_correct += calculate_accuracy(
                logits.view(-1, output_dim), batch_y.view(-1)) * batch_y.size(0)
            val_total += batch_y.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= len(valid_dataloader)
    return val_loss, val_accuracy

## 3-7. 모델 학습하기

정해진 에포크(10회)만큼 학습을 반복한다. 각 배치마다 순전파로 예측(logits)을 구하고, 실제 정답과 비교해 loss를 계산한 뒤 역전파·최적화로 가중치를 갱신한다. 에포크가 끝날 때마다 훈련·검증의 loss와 정확도를 출력하고, **검증 loss가 최소를 갱신할 때마다 체크포인트를 저장**해 가장 성능이 좋은 모델을 남긴다.

In [ ]:
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ----- 학습 -----
    train_loss = 0
    train_correct = 0
    train_total = 0

    model.train()
    for batch_X, batch_y in train_dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        logits = model(batch_X)  # 순전파
        loss = criterion(logits.view(-1, output_dim), batch_y.view(-1))

        optimizer.zero_grad()  # 역전파 & 최적화
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_correct += calculate_accuracy(
            logits.view(-1, output_dim), batch_y.view(-1)) * batch_y.size(0)
        train_total += batch_y.size(0)

    train_accuracy = train_correct / train_total
    train_loss /= len(train_dataloader)

    # ----- 검증 -----
    val_loss, val_accuracy = evaluate(model, valid_dataloader, criterion, device)

    print(f'Epoch {epoch+1}/{num_epochs}:')
    print(f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}')
    print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

    # 검증 loss가 최소일 때 체크포인트 저장
    if val_loss < best_val_loss:
        print(f'Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}. 체크포인트를 저장합니다.')
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model_checkpoint.pth')

## 3-8. 모델 로드 및 평가

저장해 둔 Best 모델을 로드해 정상적으로 불러와졌는지 검증 데이터로 확인하고, 이어서 테스트 데이터에 대해서도 정확도와 loss를 측정한다.

In [ ]:
# Best 모델 로드
model.load_state_dict(torch.load('best_model_checkpoint.pth'))
model.to(device)

# 검증 데이터 평가
val_loss, val_accuracy = evaluate(model, valid_dataloader, criterion, device)
print(f'Best model validation loss: {val_loss:.4f}')
print(f'Best model validation accuracy: {val_accuracy:.4f}')

테스트 데이터에 대해서도 같은 방식으로 평가한다.

In [ ]:
test_loss, test_accuracy = evaluate(model, test_dataloader, criterion, device)
print(f'Best model test loss: {test_loss:.4f}')
print(f'Best model test accuracy: {test_accuracy:.4f}')

## 3-9. 인퍼런스 및 테스트

실제 서비스에서는 **전처리가 전혀 안 된 임의의 텍스트**가 입력으로 들어온다. 그런 입력을 받아 예측 태그를 돌려주는 함수를 만든다. 정수→태그 변환용 `index_to_tag`를 만들고, 입력 문장을 토큰화 → 정수 인코딩 → 패딩 → 예측 순으로 처리한다.

In [ ]:
# 정수 -> 태그 매핑
index_to_tag = {}
for key, value in tag_to_index.items():
    index_to_tag[value] = key

def predict_labels(text, model, word_to_ix, index_to_tag, max_len=80):
    tokens = text.split()  # 단어 토큰화

    # 정수 인코딩 (OOV는 1=<UNK>)
    token_indices = [word_to_ix.get(token, 1) for token in tokens]

    # 패딩
    token_indices_padded = np.zeros(max_len, dtype=int)
    token_indices_padded[:len(token_indices)] = token_indices[:max_len]

    # 텐서로 변환 (배치 차원 추가)
    input_tensor = torch.tensor(token_indices_padded, dtype=torch.long).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)

    # 각 시점에서 가장 높은 인덱스를 예측으로 선택
    predicted_indices = torch.argmax(logits, dim=-1).squeeze(0).tolist()
    predicted_indices_no_pad = predicted_indices[:len(tokens)]  # 패딩 제거

    predicted_tags = [index_to_tag[index] for index in predicted_indices_no_pad]
    return predicted_tags

# 테스트 데이터 첫 샘플 (이미 토큰화된 상태)
print(X_test[0])

이 데이터는 이미 토큰화되어 있으므로, 공백으로 다시 이어 붙여 **전처리 이전 상태의 문장**을 가정해 함수에 넣는다. 예측 결과와 실제 정답을 비교해 본다.

In [ ]:
# 토큰화 이전 상태(공백으로 연결된 문장)로 복원
sample = ' '.join(X_test[0])
print(sample)
print()

predicted_tags = predict_labels(sample, model, word_to_index, index_to_tag)
print('예측 :', predicted_tags)
print('실제값 :', y_test[0])

---

이로써 14장 **시퀀스 레이블링**을 정리했다. 입력의 각 원소마다 레이블을 붙이는 **다-대-다(Many-to-Many)** 작업이라는 점, 그리고 앞뒤 문맥을 모두 참고하기 위해 **양방향 RNN(LSTM/GRU)**을 쓴다는 점이 핵심이다. 대표 작업인 **개체명 인식(NER)**을 NLTK로 먼저 맛본 뒤, **2층 양방향 LSTM**으로 데이터 로드 → Vocab → 정수 인코딩 → 패딩 → 모델링 → 학습(체크포인트 저장) → 평가 → 인퍼런스까지 전 과정을 직접 구현했다.

특히 패딩 처리에서 **`CrossEntropyLoss(ignore_index=0)`로 패딩 위치의 loss를 제외**하고, 정확도 계산에서도 **패딩을 마스킹**해 실제 토큰만 평가한 점이 태깅 작업의 중요한 디테일이다. 다음 장부터는 서브워드 토크나이저 등 더 정교한 전처리로 나아간다.